<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/Course_Work/LLM_Examples_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini LLM example

In [ ]:
# Install the official Gemini Python SDK
!pip -q install -U google-genai matplotlib
!pip -q install pandas==2.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


In [ ]:
# Imports
import os
import json
import textwrap
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import userdata
from google import genai

In [ ]:
# Read API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY not found in Colab Secrets. "
        "Please add it using the key icon in the left sidebar."
    )

# Create Gemini client
client = genai.Client(api_key=api_key)

# Pick a fast, low-cost text model
MODEL_NAME = "gemini-2.5-flash"

print("Gemini client created successfully.")
print("Model selected:", MODEL_NAME)

Gemini client created successfully.
Model selected: gemini-2.5-flash


In [ ]:
def generate_text(
    prompt: str,
    model: str = MODEL_NAME,
    temperature: float = 0.7,
    top_p: float = 0.95,
    top_k: int | None = None,
    max_output_tokens: int = 1000,
):
    """Generate text with Gemini and return the plain text response."""

    config = {
        "temperature": temperature,
        "top_p": top_p,
        "max_output_tokens": max_output_tokens,
    }

    if top_k is not None:
        config["top_k"] = top_k

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=config,
    )
    return response.text


def count_tokens_for_text(text: str, model: str = MODEL_NAME):
    """Return token count for a given text."""
    token_info = client.models.count_tokens(
        model=model,
        contents=text,
    )
    return token_info.total_tokens

In [ ]:
prompt_1 = """Explain the working principle of a centrifugal pump in simple terms
for a first-year engineering learner. Use 2 bullet points."""

response_1 = generate_text(prompt_1)
print(response_1)

Here's the working principle of a centrifugal pump:

*   **Energy Impartation:** A spinning component called an **impeller** (with internal vanes) receives fluid at its center. As the impeller rotates at high speed, it throws the fluid outwards towards its edges due to **centrifugal force**, imparting high velocity and thus kinetic energy to the fluid.
*   **Energy Conversion:** The high-velocity fluid then enters a surrounding, gradually expanding casing (called a **volute** or **diffuser**). As the flow area increases, the fluid slows down, and its kinetic energy is efficiently converted into pressure energy, creating the head required to push the fluid through the piping system.


In [ ]:
print("Prompt token count:", count_tokens_for_text(prompt_1))
print("Response token count:", count_tokens_for_text(response_1))

Prompt token count: 27
Response token count: 143


In [ ]:
prompts = {
    "generic": "Explain a centrifugal pump.",
    "structured": "Explain the working principle of a centrifugal pump in 5 bullet points.",
    "role_based": (
        "You are a trainer for mechanical engineers. "
        "Explain the working principle of a centrifugal pump using simple industrial language."
    ),
    "task_specific": (
        "Explain the working principle of a centrifugal pump and then list 3 common maintenance issues."
    ),
}

results = []
for name, p in prompts.items():
    out = generate_text(p, temperature=0.7)
    results.append({
        "prompt_type": name,
        "prompt": p,
        "response": out,
        "prompt_tokens": count_tokens_for_text(p),
        "response_tokens": count_tokens_for_text(out),
    })

df_prompts = pd.DataFrame(results)
df_prompts[["prompt_type", "prompt_tokens", "response_tokens"]]

,prompt_type,prompt_tokens,response_tokens
0,generic,6,40
1,structured,15,41
2,role_based,22,39
3,task_specific,18,43
